# Chapter 7: Fine-tuning a GPT to follow instructions

Chapter 6 took the pretrained GPT and taught it a closed question: read a whole message, then pick a label from a fixed list. The head changed, and the model stopped writing tokens. This chapter is the other kind of fine-tuning. The model keeps the language-model head from Chapter 4 and keeps predicting the next token. What changes is the **text we train on**, and **which of those tokens the loss is allowed to grade**.

An instruction example is a small task with a known good answer, not a paragraph cut out of a book. Each record has three fields:

- **instruction** — the task itself. In the sample below, "identify the programming language of this snippet".
- **input** — extra material the task needs. Here that is the snippet `print('Hello, Yash')`. Some tasks have no input at all, and the template has to survive that.
- **output** — the answer we want written, `Python` in the sample. This field is for training. When we later ask the model a new question, we leave it blank.

Pretraining, classification, and instruction tuning get mixed up because people call all three "training". Only the last two are fine-tuning, and they specialise the model in different ways:

| | Pretraining (Chapter 4) | Classification (Chapter 6) | Instruction tuning (this chapter) |
|---|---|---|---|
| **Question the model answers** | "What token comes next?" | "Which label is this whole text?" | "What token comes next in the answer?" |
| **Output size** | `vocab_size` scores at every position | `num_classes` scores for the whole sequence | `vocab_size` scores at every position, same as pretraining |
| **Target** | a token ID at every position | one class ID per example | token IDs of the **response**; the prompt is marked `-100` |
| **Data** | raw text | a labelled spreadsheet | instruction / input / output records |
| **What changes in the network** | the whole model | a new linear head, body frozen | the same model; the template and the loss mask do the specialising |

A pretrained GPT already knows how to continue text. Hand it "Identify the language" with no further training and it may continue with another question, a blog sentence, or whatever was common next to those words on the internet. Instruction tuning teaches a more specific habit: when the text is wrapped in the template below, the continuation after the response heading should be the answer to the task. Chat-style behaviour starts here. The architecture is the one you already built.

Here is the path we will follow:

- **A prompt template**: one fixed layout for the instruction, the optional input, and the response, so every example looks alike
- **An instruction dataset**: encode the full example, then mark the prompt tokens and the padding as "do not grade" with `-100`
- **Masked cross-entropy**: Chapter 4's loss, plus `ignore_index=-100`, so masked positions add nothing to the number and send no gradient
- **A response generator**: feed the prompt, let the model write one token at a time, and cut the answer back out of the decoded string

A quick reminder about the shapes. They look like pretraining again, and nothing like the `(batch, num_classes)` head from Chapter 6:

- **input token IDs** are length `max_length` per example, and shape `(batch, max_length)` once a loader stacks them
- **targets** have that same shape, with `-100` in every position we refuse to learn from
- **logits**, from the original GPT, are still `(batch, max_length, vocab_size)`
- the **head stays the language-model head**. This chapter does not add a classifier

## Step 1: Formatting prompt templates

A model cannot see a Python dict. It sees a string of tokens. If one training row says `Task:`, the next says `Q:`, and a third is a bare sentence, there is no stable place where the request ends and the answer begins. `format_input` removes that freedom. Every record is poured into the same mould. The wording is the Alpaca-style wrapper used by a lot of open instruction datasets: a one-line preamble, then `###` headings.

The function builds **only the prompt**, the part a user would type. The answer is glued on afterwards, in the lines under the function.

1. Three f-strings are concatenated into `instruction_text`. They are always there:
   - a preamble telling the model that an instruction follows and that it should write a response
   - a blank line
   - the heading `### Instruction:` and then `entry["instruction"]`
2. The two preamble f-strings are joined with nothing between them, so the printed line reads `task.Write` with no space. That run of characters is what the tokenizer will see. The template is the string in the source, including this.
3. `entry.get("input")` looks up the optional context. `.get` returns `None` when the key is missing, and both `None` and `""` are skipped, so a task with no extra material stays a two-part prompt. When the field has text, the function appends a blank line, the heading `### input:`, and that text.
4. The return value is the prompt. `format_input` itself never writes the answer. That split is the whole point of the next cell: the prompt is what we mask, and the answer is what we train on.

The rest of the cell builds one toy record and prints the full training string.

- `instruction`: identify the language used in a snippet (the sample spells it `Indentify`)
- `input`: `print('Hello, Yash')`
- `output`: `Python`

`full_entry` takes the prompt and appends a blank line, the heading `### Response:`, and `sample_data["output"]`. One training example is therefore a single string with three visible regions: the preamble, the task, and the gold answer. The printout under the cell is that string. Read it the way the model will: left to right, one token after another.

Nothing in those characters says "skip the first half during training". The string is just text. Step 2 tokenizes it and writes `-100` over the prompt positions, and that is the actual instruction to the loss.

When you later ask the model something new, you call `format_input` the same way and stop at the response heading, with the answer left empty for the model to fill. The headings are part of the method. A model that studied `### Instruction:` has no reason to follow a prompt that suddenly says `Question:`.

In [1]:
def format_input(entry):
    instruction_text=(
        f"Below is an instruction that describe a task."
        f"Write a response that appropriately completes the request.\n\n"
        f"### Instruction:\n{entry['instruction']}")
    if entry.get("input"):
        instruction_text+=f"\n\n### input:\n{entry['input']}"
    return instruction_text
sample_data={
    "instruction": "Indentify the primary programming language used in this snippet.",
    "input": "print('Hello, Yash')",
    "output":"Python"}

formatted_prompt = format_input(sample_data)

full_entry= f"{formatted_prompt}\n\n### Response:\n{sample_data['output']}"

print(full_entry)

Below is an instruction that describe a task.Write a response that appropriately completes the request.

### Instruction:
Indentify the primary programming language used in this snippet.

### input:
print('Hello, Yash')

### Response:
Python


## Step 2: An instruction dataset with masked targets

`InstructionDataset` turns a list of instruction records into the tensors a training step can eat. It is a PyTorch `Dataset`, the same contract as `SpamDataset` in Chapter 6:

- `__len__` returns how many examples were stored
- `__getitem__(idx)` returns example number `idx`

What comes back is a different kind of pair. Chapter 6 returned `(token IDs, one class ID)`. This class returns `(token IDs, token IDs)`, and the second tensor is full of `-100` wherever the model must not be graded.

`__init__` walks `data_list` once. For every record it appends one input tensor to `self.input_ids` and one target tensor to `self.target_ids`. `max_length` defaults to 1,024, the context length of the GPT-2 loaded in Chapter 4.

### From one record to a masked pair

1. **`format_input(entry)`** builds the prompt from Step 1: preamble, instruction, and the input block when the record has one. The answer is not in this string.
2. **`full_text`** appends the response heading and `entry["output"]`. In this cell that heading is written `###Response:` with no space after the hashes. The sample `full_entry` in the previous cell used `### Response:` with a space. The tokenizer will learn whichever characters are actually in the training string, so it is worth noticing that these two cells do not spell the heading the same way. The generator in Step 4 has to use the spelling the dataset trained on.
3. **`tokenizer.encode`** is called twice. `tokenized_full` is the whole example, prompt plus answer. `tokenized_prompt` is only the request. The length of that second list is how the code finds where the answer starts.
4. **`tokenized_full` is cut with `[:max_length]`.** Anything past the context window cannot enter the model, so the row is capped there.
5. **The target is stitched from two pieces.**
   - `[-100] * len(tokenized_prompt)` writes one ignore-label per prompt token
   - `tokenized_full[len(tokenized_prompt):]` copies the real token IDs of whatever follows, which is the response (and is already capped by the slice in step 4)

The input keeps the prompt. The model has to be able to read the question. The target wipes it:

```
input:   [ prompt tokens | response tokens ]
target:  [ -100  -100 .. | response token IDs ]
```

Encoding the prompt and the full text as two separate calls is the straightforward way to measure that split. It lines up when `tokenized_full` really does begin with `tokenized_prompt`. Byte-pair encoding sometimes merges the last token of the prompt with the first token of the response, so the two lists can disagree by a token at the boundary. Before a long training run it is worth checking that the prompt IDs are a prefix of the full IDs. The mask width in this cell is `len(tokenized_prompt)`, and it assumes that prefix.

6. **Right-padding.** When the row is shorter than `max_length`, the input is filled with `tokenizer.eot_token` and the target is filled with more `-100`. This is the same end-of-text pad Chapter 6 used (`<|endoftext|>`) rather than a raw `0`. The extra pad positions are filler past the end of the answer, so their labels stay ignored. Both tensors are then length `max_length`, which is what a batch needs: every row in a batch has to be the same width.

`__getitem__` returns the pair at `idx`. A `DataLoader` can stack those pairs into `input_ids` of shape `(batch, max_length)` and `target_ids` of the same shape.

### Why -100, and why only on the prompt

`-100` is the usual "skip me" label. It is not a token ID. GPT-2's vocabulary runs from 0 to 50,256, so `-100` cannot be mistaken for a real class. The next cell tells cross-entropy to ignore exactly this value.

The mask is the difference between "continue this text" and "answer this request". If the prompt tokens were ordinary targets, most of the loss would be the model copying the user's instruction back. We already have that text; we wrote it into the prompt. The skill we want is the continuation after the response heading. Positions labelled `-100` are still *read* by causal attention. They are simply not graded.

This is also the clean split with Chapter 6. Classification replaced the 50,257-way head and pooled the last token into a class score. Instruction tuning keeps that head. The specialisation lives in the targets.

In [4]:
import torch 
from torch.utils.data import Dataset 

class InstructionDataset(Dataset):
    def __init__(self, data_list, tokenizer, max_length=1024):
        self.input_ids=[]
        self.target_ids=[]
        for entry in data_list:
            formatted_prompt = format_input(entry)
            full_text= f"{formatted_prompt}\n\n###Response:\n{entry['output']}"

            tokenized_full = tokenizer.encode(full_text)
            tokenized_prompt = tokenizer.encode(formatted_prompt)
            tokenized_full = tokenized_full[:max_length]
            tokenized_prompt = tokenized_prompt[:len(tokenized_prompt)]
            target_labels=[-100]*len(tokenized_prompt)
            target_labels+= tokenized_full[len(tokenized_prompt):]
            padding_len = max_length - len(tokenized_full)
            if padding_len>0:
                tokenized_full+=[tokenizer.eot_token]*padding_len
                target_labels+=[-100]*padding_len 

            self.input_ids.append(torch.tensor(tokenized_full))
            self.target_ids.append(torch.tensor(target_labels))
    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

## Step 3: Masked cross-entropy

Chapter 4 graded every position. Flatten the logits from `(batch, tokens, vocab_size)` to `(batch * tokens, vocab_size)`, flatten the targets to match, and call cross-entropy. That is still the right loss. An instruction batch simply contains positions that must not count: the prompt, and the padding. Both already carry the label `-100` from `InstructionDataset`.

`nn.CrossEntropyLoss(ignore_index=-100)` is that loss with one extra rule. Any target equal to `-100` is dropped before the average. It adds nothing to the number, and it sends no gradient. The response tokens, whose targets are real vocabulary IDs, are scored exactly as in pretraining:

$$\mathcal{L} = -\frac{1}{|A|} \sum_{t \in A} \log \, \text{softmax}(z_t)_{y_t}$$

$z_t$ is the vector of `vocab_size` logits at position $t$, $y_t$ is the target token ID there, and $A$ is the set of positions where $y_t \neq -100$. On a typical row, $A$ is only the answer. A long prompt and a one-word answer like `Python` means almost the entire row is ignored, and the loss is decided by the few tokens of the response. That imbalance is the method, not an accident. We want the gradient spent on the answer.

The cell builds this object and prints a confirmation. It does not run a model. The print is a status line: the loss you will train with is the one that honours the mask. When a training step does run, the call is the Chapter 4 call, pointed at `masked_loss_fn`:

```
logits = model(input_batch)          # (batch, max_length, vocab_size)
loss = masked_loss_fn(
    logits.flatten(0, 1),            # (batch * max_length, vocab_size)
    target_batch.flatten()           # (batch * max_length,)
)
```

`flatten(0, 1)` is the same trick as before. Every position becomes its own classification problem over the vocabulary. `ignore_index` then throws away the rows whose label is `-100`, so a batch can hold thousands of positions while only the answer tokens move the weights.

Causal attention still runs over the whole sequence, prompt included. The model is allowed to *read* the instruction. It is only graded on what it writes after the response heading. The mask changes the loss, not the forward pass.

In [5]:
import torch 
import torch.nn as nn

masked_loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
print("Loss Configuration status: ignore_index=-100 is actively locked!")

Loss Configuration status: ignore_index=-100 is actively locked!


## Step 4: Generating a response

Steps 2 and 3 are the training side: a batch of masked targets, and a loss that skips `-100`. Wired into the Chapter 4 loop (`zero_grad`, forward, `masked_loss_fn`, `backward`, `step`), they would update the weights. The function in the next cell is the other half. Given a new instruction, write an answer. There is no loss and no optimizer here. The model is only being read.

`generate_instruction_response` is greedy decoding, the same idea as `generate_text_simple` from the pretraining notebook, wrapped in the instruction template.

1. **`model.eval()`** turns dropout off. The same prompt should not wander just because random units were masked on this pass.
2. **The prompt is rebuilt with `format_input`.** `instruction` and `input_text` are packed into the dict that template expects. The default `input_text=None` leaves the `### input:` block out, because `entry.get("input")` is then empty. The cell appends `\n\n###Response:\n` and stops. The string ends on the response heading. The answer is what the loop is about to write, which is the same place the gold `output` sat during training.
3. **Encode, then add a batch dimension.** `tokenizer.encode` returns a Python list of IDs. `torch.tensor(...).unsqueeze(0)` turns it into shape `(1, prompt_len)`: one sequence, batch size 1, which is what `model` expects.
4. **Write tokens one at a time, up to `max_new_tokens` (50).** The loop sits inside `torch.no_grad()`, because we are not building a graph:
   - `model(input_ids)` returns logits of shape `(1, current_len, vocab_size)`
   - `logits[:, -1, :]` keeps only the last position. Causal attention means that position has been allowed to see the entire prompt and every token written so far
   - `torch.argmax(..., dim=1, keepdim=True)` picks the single highest score. This is greedy decoding: no sampling, so a given prompt produces a given continuation
   - `torch.cat` appends that ID along the sequence dimension, and the next pass sees a longer input
   - if the chosen ID equals `tokenizer.eot_token`, the model has emitted GPT-2's end-of-text token and the loop breaks early
5. **Decode, then cut.** `tokenizer.decode` turns the full ID list back into text, prompt included. `.split("### Response:\n")[-1]` keeps the text after the last match of that heading, and `.strip()` drops the surrounding whitespace. The value returned to the caller is meant to be the answer on its own.

Read the last three lines of the cell against the string built just above them, before you call the function. The prompt is closed with `###Response:` (no space after the hashes). The `.split` looks for `### Response:` (with a space). The dataset cell uses the no-space spelling too, while the sample `full_entry` in Step 1 uses the spaced one. The extracted piece is stored in `reponse_part`, and the `return` reads `response_part`. Those are two different names, so as the cell stands the return does not see the variable the split just created.

The procedure in the five steps above is what the function is written to do. For the cut to land on the answer, the heading in the generator, the heading in `InstructionDataset`, and the string passed to `.split` have to be the same characters, and the name that is returned has to be the name that was assigned.

In [ ]:
def generate_instruction_response(model, tokenizer, instruction, input_text=None, max_new_tokens=50):
    model.eval()
    entry={"instruction": instruction, "input": input_text}
    formatted_prompt= format_input(entry)+"\n\n###Response:\n"

    input_ids= torch.tensor(tokenizer.encode(formatted_prompt)).unsqueeze(0)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits=model(input_ids)
            next_token=torch.argmax(logits[:, -1, :], dim=1, keepdim=True)
            input_ids= torch.cat((input_ids, next_token), dim=-1)

            if next_token.item()==tokenizer.eot_token:
                break
    decoded_text= tokenizer.decode(input_ids[0].tolist())

    reponse_part= decoded_text.split("### Response:\n")[-1]
    return response_part.strip()

## Summary

Pretraining taught the model to continue text. Chapter 6 taught it to pick a label. This chapter teaches it to answer a request, using the same next-token head it already had.

- **Instruction fine-tuning** keeps the language-model head. The model still scores every vocabulary token at every position. The new behaviour comes from the examples and from a mask, not from a new layer.
- **`format_input`** pours every record into one template: a preamble, `### Instruction:`, an optional `### input:`, and then a response heading plus the gold answer. A new question has to be wrapped in that same template.
- **`InstructionDataset`** encodes the full string, writes `-100` over the prompt and the padding, and leaves real token IDs only on the response. The model reads the question and is graded on the answer.
- **`CrossEntropyLoss(ignore_index=-100)`** is Chapter 4's loss with those positions removed from the average. After the usual `flatten`, only the answer tokens move the weights.
- **`generate_instruction_response`** feeds the prompt up through the response heading and appends the greedy next token until end-of-text or `max_new_tokens`. Decoding the whole string and splitting on the response heading is how the answer is pulled back out.

The headings are part of the method. The training string, the string the generator builds, and the string `.split` searches for all have to be the same characters, or the model is answering a different format from the one it studied.